In [1]:
import numpy as np
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
import itertools as it
import scipy.stats.qmc as ssq
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space.space import Real, Integer
from sklearn.preprocessing import MinMaxScaler, StandardScaler

np.set_printoptions(threshold=100_000)
X = pd.read_csv(r"C:\Users\dwigh\OneDrive\Desktop\Emissions Uncertainty on Antarctic Instability\ensemble_output\results\default\parameters.csv")

Y = pd.read_csv(r"C:\Users\dwigh\OneDrive\Desktop\Emissions Uncertainty on Antarctic Instability\ensemble_output\results\default\gmslr.csv")
Y = Y.mean(axis=1).rename('output')
obj = X.join(Y, how='left')
obj.sort_values(by='t_peak',ascending=True, inplace=True)


In [2]:
# obj.t_peak = pd.to_datetime(obj.t_peak, format='%Y')
obj.set_index('t_peak',drop=False, inplace=True)
obj

,gamma_g,t_peak,gamma_d,sd_temp,sd_ocean_heat,sd_glaciers,sd_greenland,sd_antarctic,sd_gmsl,rho_temperature,...,antarctic_kappa,antarctic_flow0,antarctic_runoff_height0,antarctic_c,antarctic_bed_height0,antarctic_slope,antarctic_lambda,antarctic_temp_threshold,lw_random_sample,output
t_peak,,,,,,,,,,,,,,,,,,,,,
2030.0,0.005979,2030.0,0.091974,0.074738,2.503238,0.000198,0.000232,0.000383,0.000405,0.467965,...,0.072566,1.072699,1240.514180,79.039386,762.770179,0.000707,0.008297,-15.740854,0.000261,0.430777
2030.0,0.009668,2030.0,0.081546,0.081082,1.851315,0.000132,0.000251,0.000502,0.001941,0.369884,...,0.076086,1.323465,1014.354512,126.821100,807.992695,0.000624,0.013372,-15.546083,0.000092,0.347866
2030.0,0.002629,2030.0,0.149528,0.082100,1.756363,0.000159,0.000247,0.000475,0.002613,0.506711,...,0.059001,1.328616,1737.814817,119.263273,809.343880,0.000708,0.006230,-15.540414,0.000242,0.365387
2030.0,0.003917,2030.0,0.081056,0.087278,2.841193,0.000082,0.000239,0.000466,0.001685,0.355840,...,0.054435,0.993865,1136.483270,140.548806,816.925800,0.000684,0.006855,-15.453124,0.000164,0.390832
2030.0,0.008057,2030.0,0.112075,0.077741,1.353660,0.000495,0.000276,0.000340,0.001981,0.538458,...,0.055346,1.470483,991.781103,97.797643,807.736344,0.000556,0.010325,-14.959565,0.000641,0.491800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2167.0,0.013229,2167.0,0.093814,0.080127,1.034033,0.000253,0.000202,0.000538,0.000763,0.518355,...,0.069314,1.460019,1379.251183,117.385686,797.296496,0.000668,0.011739,-16.190533,0.000739,3.135588
2169.0,0.003130,2169.0,0.022636,0.079961,2.088501,0.000069,0.000217,0.000380,0.001736,0.665597,...,0.065737,1.516385,1673.690577,136.186139,787.740252,0.000562,0.012827,-16.224729,0.000183,2.118476
2171.0,0.004158,2171.0,0.120120,0.078078,0.375789,0.000049,0.000264,0.000451,0.002437,0.763427,...,0.080312,0.812621,1340.939748,76.307069,798.575153,0.000548,0.009879,-15.795974,0.000354,2.240000


In [ ]:
color_pal = sns.color_palette()
plt.style.use('fivethirtyeight')
obj.output.plot(figsize=(15,5),color=color_pal[0], kind='kde')
plt.show()

In [3]:
def add_lags(data: pd.DataFrame):
    new_data = data.copy()
    index = pd.to_datetime(new_data.t_peak, format='%Y')
    target_map = new_data['output'].to_dict()
    lag1 =  (index - pd.Timedelta('365 days')).dt.year
    lag2 = (index - pd.Timedelta('730 days')).dt.year
    lag3 = (index - pd.Timedelta('1095 days')).dt.year
    new_data['lag1'] = lag1.map(target_map)
    new_data['lag2'] = lag2.map(target_map)
    new_data['lag3'] = lag3.map(target_map)
    # new_data['lag4'] = (new_data.index - pd.Timedelta('1460 days')).map(target_map)
    return new_data

In [4]:
new_obj = add_lags(obj)
new_obj.tail()

,gamma_g,t_peak,gamma_d,sd_temp,sd_ocean_heat,sd_glaciers,sd_greenland,sd_antarctic,sd_gmsl,rho_temperature,...,antarctic_c,antarctic_bed_height0,antarctic_slope,antarctic_lambda,antarctic_temp_threshold,lw_random_sample,output,lag1,lag2,lag3
t_peak,,,,,,,,,,,,,,,,,,,,,
2167.0,0.013229,2167.0,0.093814,0.080127,1.034033,0.000253,0.000202,0.000538,0.000763,0.518355,...,117.385686,797.296496,0.000668,0.011739,-16.190533,0.000739,3.135588,2.242828,2.037348,2.453863
2169.0,0.003130,2169.0,0.022636,0.079961,2.088501,0.000069,0.000217,0.000380,0.001736,0.665597,...,136.186139,787.740252,0.000562,0.012827,-16.224729,0.000183,2.118476,NaN,3.135588,2.242828
2171.0,0.004158,2171.0,0.120120,0.078078,0.375789,0.000049,0.000264,0.000451,0.002437,0.763427,...,76.307069,798.575153,0.000548,0.009879,-15.795974,0.000354,2.240000,NaN,2.118476,NaN
2172.0,0.007347,2172.0,0.049480,0.072067,2.871035,0.000278,0.000222,0.000539,0.001515,0.628720,...,120.324753,811.646750,0.000652,0.014322,-15.887230,0.000333,2.066674,2.240000,NaN,2.118476
2178.0,0.007347,2178.0,0.163910,0.076930,1.829964,0.000510,0.000244,0.000358,0.000564,0.369075,...,104.880124,799.678981,0.000679,0.006349,-16.138989,0.000366,1.507992,NaN,NaN,NaN


In [5]:
new_obj1 = new_obj.copy()
output = new_obj1.pop('output')
new_obj1.insert(56, 'output', output)

In [ ]:
val = TimeSeriesSplit(n_splits=4, max_train_size=60_000)
features = new_obj1.iloc[:,:-1].columns
target = new_obj1.iloc[:,-1].name
scores = {'Train Error':list(), 'Test Error': list()}
folds = 0
for train_indx, val_indx in val.split(new_obj1):
    train = new_obj1.iloc[train_indx]
    test = new_obj1.iloc[val_indx]

    X_train = train[features]
    Y_train = train[target]
    X_test = test[features]
    Y_test = test[target]
    RF = RandomForestRegressor(n_estimators=300, n_jobs=10, random_state=0)
    RF.fit(X_train, Y_train)
    y_pred = RF.predict(X_train)
    y_tpred = RF.predict(X_test)
    val_error = np.square(Y_test - y_tpred).mean()
    train_error = np.square(Y_train - y_pred).mean()
    print(f'[{folds}]Training MSE:\t{train_error} Validation MSE:\t{val_error}')
    scores['Train Error'].append(train_error)
    scores['Test Error'].append(val_error)
    folds+=1